# Food Recognition & 3D Plate Scanning Research Pipeline
### Google Colab GPU Backup Training: 43-Class Indian Food Classification (15,404 Images)

**Source of Truth Repository:** `https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git`  
**Primary Platform:** Kaggle GPU is primary. Use this Colab notebook as a backup runtime.

--- 
### ⚠️ Colab Persistence & Compliance Notice
1. **Ephemeral Local Storage:** `/content` is destroyed when the Colab runtime disconnects. To ensure permanent checkpoint persistence, optional Google Drive mounting is provided below.
2. **43-Class Pipeline:** Consumes the validated 15,404-image Enhanced Indian Food dataset across 43 classes.
3. **Zero Hardcoded Secrets:** Never paste GitHub tokens or credentials into public cells.

### Step 1: Clone GitHub Repository (Source of Truth)

In [ ]:
import os
import sys
import subprocess

REPO_URL = "https://github.com/VaibhavBajpaij/food-plat-scanner-ai-model.git"
REPO_DIR = "/content/food-plat-scanner-ai-model"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}. Pulling latest commits...")
    subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
else:
    print(f"Cloning source of truth repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"Current Working Directory: {os.getcwd()}")

### Step 2: Hardware, GPU & Disk Storage Environment Audit

In [ ]:
import shutil
import urllib.request
import torch

print("=" * 85)
print("GOOGLE COLAB RUNTIME & HARDWARE AUDIT")
print("=" * 85)
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\nPyTorch Version:         {torch.__version__}")
print(f"CUDA Available:          {cuda_available}")
if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU Accelerator:  {device_name}")
    print(f"Total Dedicated VRAM:    {total_mem_gb:.2f} GB")
else:
    print("WARNING: GPU is not active! Navigate to Runtime -> Change runtime type -> T4 GPU.")

total_disk, used_disk, free_disk = shutil.disk_usage("/content")
print(f"Disk Space (/content): Total={total_disk/(1024**3):.1f} GB | Free={free_disk/(1024**3):.1f} GB")
print("=" * 85)

### Step 3 (Optional): Mount Google Drive for Persistent Storage

In [ ]:
# Mount Google Drive if persistent cloud saving is desired
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_EXPORT_DIR = "/content/drive/MyDrive/food_model_checkpoints"
    os.makedirs(DRIVE_EXPORT_DIR, exist_ok=True)
    print(f"Google Drive mounted successfully. Checkpoints will sync to: {DRIVE_EXPORT_DIR}")
except Exception as e:
    print(f"Skipping Google Drive mount (or running outside Colab UI): {e}")

### Step 4: Install Required Dependencies

In [ ]:
!pip install -q pyarrow fastparquet Pillow torchvision
print("Dependencies installed successfully.")

### Step 5: Acquire Approved Parquet Shards (15,404 Images)

In [ ]:
from src.data.resumable_downloader import EnhancedIndianFoodAdapter

adapter = EnhancedIndianFoodAdapter(staging_dir="/content/datasets/raw/sohlhealth_indian_food")
success = adapter.acquire(resume=True)
assert success, "Dataset acquisition failed!"
print("Enhanced Indian Food Parquet shards downloaded.")

### Step 6: Parquet Extraction & 43-Class Manifest Preparation

In [ ]:
from src.data.parquet_food_loader import extract_and_prepare_43class_manifest

manifest_result = extract_and_prepare_43class_manifest(
    parquet_dir="/content/datasets/raw/sohlhealth_indian_food",
    output_image_dir="/content/datasets/processed/indian_food_43class",
    output_manifest_path="/content/data_manifests/indian_food_43class_manifest.json"
)
assert manifest_result["classes_count"] == 43
print(f"Verified {manifest_result['total_images']:,} images across 43 classes on Colab.")

### Step 7: Run Unit Tests & 43-Class Smoke Test

In [ ]:
!python -m src.utils.run_unit_tests

### Step 8: Execute Staged Transfer Learning (43 Classes)

In [ ]:
from src.classification.train_kaggle import train_indian_43class_model

report = train_indian_43class_model(
    manifest_path="/content/data_manifests/indian_food_43class_manifest.json",
    epochs_head=5,
    epochs_finetune=10,
    batch_size=32,
    lr_head=1e-3,
    lr_finetune=1e-4,
    seed=42,
    num_workers=2
)

### Step 9: Package & Persist Artifacts

In [ ]:
import shutil
import zipfile

zip_dest = "/content/colab_indian_43class_artifacts.zip"
with zipfile.ZipFile(zip_dest, 'w', zipfile.ZIP_DEFLATED) as zipf:
    ckpt_path = "/content/models/mobilenet_v3_indian_43class_best.pth"
    if os.path.exists(ckpt_path):
        zipf.write(ckpt_path, arcname="mobilenet_v3_indian_43class_best.pth")
    rep_path = "/content/outputs/kaggle_indian_43class_training_report.json"
    if os.path.exists(rep_path):
        zipf.write(rep_path, arcname="kaggle_indian_43class_training_report.json")
    man_path = "/content/data_manifests/indian_food_43class_manifest.json"
    if os.path.exists(man_path):
        zipf.write(man_path, arcname="indian_food_43class_manifest.json")

print(f"Artifact bundle created at {zip_dest} ({os.path.getsize(zip_dest)/(1024**2):.2f} MB).")
if os.path.exists("/content/drive/MyDrive/food_model_checkpoints"):
    shutil.copy(zip_dest, "/content/drive/MyDrive/food_model_checkpoints/")
    print("Artifact bundle synced to persistent Google Drive storage.")